# Notebook 02 — Preprocesamiento

**Entrada:** `loan_recodificado.parquet`  
**Salida:** `loan_procesado.parquet`, `splits.pkl`

## Contenido
1. Carga del dataset recodificado
2. Eliminación de variables no útiles
3. Tratamiento de valores faltantes
4. Selección de variables con Information Value (IV)
5. Transformación WOE
6. Manejo del desbalance de clases
7. División train / validation / test

## 1. Configuración y carga de datos

In [ ]:
import numpy             as np
import pandas            as pd
import matplotlib.pyplot as plt
import seaborn           as sns
import scorecardpy       as sc
import warnings
import os

from sklearn.model_selection  import train_test_split
from imblearn.under_sampling  import RandomUnderSampler
from imblearn.over_sampling   import SMOTE

warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 50)

plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid', font_scale=1.1)

# Rutas del proyecto
DATA_PATH    = os.path.join('..', 'data')
FIGURES_PATH = os.path.join('..', 'figures')
OUTPUT_PATH  = os.path.join('..', 'outputs')

PALETA_CLASES = {0: '#2196F3', 1: '#F44336'}

In [ ]:
df = pd.read_parquet(os.path.join(DATA_PATH, 'loan_recodificado.parquet'))

print(f'Dataset cargado: {df.shape[0]:,} filas  |  {df.shape[1]} columnas')

## 2. Eliminación de variables no útiles

Antes de modelar, es fundamental depurar el conjunto de variables disponibles.
Se eliminan seis categorías de variables por razones distintas:

### 2.1 Identificadores (`id`, `member_id`, `url`)

Estas variables son etiquetas administrativas sin información sobre el comportamiento
financiero del solicitante. Incluirlas introduciría ruido sin beneficio predictivo.

### 2.2 Variables con alto porcentaje de valores faltantes (> 40%)

Las variables con más del 40% de nulos no representan de forma confiable la población.
Cualquier imputación introduciría más ruido que información real.
El umbral del 40% es convención en la literatura de riesgo de crédito
(Siddiqi, 2006; Thomas et al., 2002).

### 2.3 Variables con varianza cero (`policy_code`)

Una variable constante no aporta ningún poder discriminante.

### 2.4 Variables de alta cardinalidad (`zip_code`, `title`, `emp_title`)

La alta cardinalidad genera dispersión y riesgo de sobreajuste.

### 2.5 Data leakage — Variables posteriores al evento

Esta es la categoría más crítica. Variables como `recoveries`, `total_pymnt` o 
`collection_recovery_fee` solo toman valores distintos de cero cuando el cliente 
ya incumplió. Incluirlas genera un optimismo inflado en las métricas (Baesens et al., 2016).

| Variable | Razón |
|---|---|
| `total_pymnt`, `total_pymnt_inv` | Monto total ya pagado |
| `total_rec_prncp`, `total_rec_int` | Capital e intereses ya recaudados |
| `recoveries`, `collection_recovery_fee` | Montos de recuperación post-incumplimiento |
| `total_rec_late_fee` | Comisiones por mora |
| `last_pymnt_amnt`, `last_pymnt_d` | Último pago realizado |
| `next_pymnt_d` | Fecha del próximo pago |
| `out_prncp`, `out_prncp_inv` | Capital pendiente |

### 2.6 Variable original `loan_status`

La variable `loan_status` es la fuente directa de `default`. Mantenerla 
sería el caso más extremo de data leakage.

### Referencias

- Siddiqi, N. (2006). *Credit Risk Scorecards*. Wiley.
- Thomas, L.C., Edelman, D.B., & Crook, J.N. (2002). *Credit Scoring and its Applications*. SIAM.
- Baesens, B., Roesch, D., & Scheule, H. (2016). *Credit Risk Analytics*. Wiley.

In [ ]:
# --- Variables a eliminar ---

# 1. Identificadores
cols_identificadores = ['id', 'member_id', 'url']

# 2. Alto porcentaje de nulos (>40%)
cols_alto_nulo = [col for col in df.columns if df[col].isnull().mean() > 0.40]

# 3. Varianza cero (un solo valor unico)
cols_varianza_cero = [col for col in df.columns if df[col].nunique() <= 1]

# 4. Alta cardinalidad
cols_alta_cardinalidad = ['zip_code', 'title', 'emp_title']

# 5. Data leakage: informacion posterior al evento de incumplimiento
cols_leakage = [
    'total_pymnt', 'total_pymnt_inv',
    'total_rec_prncp', 'total_rec_int',
    'recoveries', 'collection_recovery_fee',
    'total_rec_late_fee', 'last_pymnt_amnt',
    'last_pymnt_d', 'next_pymnt_d',
    'out_prncp', 'out_prncp_inv'
]

# 6. Variable original reemplazada por 'default'
cols_origen = ['loan_status']

# Union de todas las columnas a eliminar
cols_eliminar = list(set(
    cols_identificadores  +
    cols_alto_nulo        +
    cols_varianza_cero    +
    cols_alta_cardinalidad +
    cols_leakage          +
    cols_origen
))

cols_eliminar = [c for c in cols_eliminar if c in df.columns]
df.drop(columns=cols_eliminar, inplace=True)

print(f'Variables eliminadas: {len(cols_eliminar)}')
print(f'  Identificadores       : {len([c for c in cols_identificadores if c in cols_eliminar])}')
print(f'  Alto % nulos (>40%)   : {len([c for c in cols_alto_nulo if c in cols_eliminar])}')
print(f'  Varianza cero         : {len([c for c in cols_varianza_cero if c in cols_eliminar])}')
print(f'  Alta cardinalidad     : {len([c for c in cols_alta_cardinalidad if c in cols_eliminar])}')
print(f'  Data leakage          : {len([c for c in cols_leakage if c in cols_eliminar])}')
print(f'\nVariables restantes: {df.shape[1]}')

In [ ]:
# Conversion de variables de fecha a escala numerica lineal
# Las fechas en formato 'Jan-2015' se convierten a meses desde una fecha de
# referencia fija, garantizando una escala lineal uniforme.

FECHA_ORIGEN = pd.Timestamp('2000-01-01')

vars_fecha = ['issue_d', 'earliest_cr_line', 'last_credit_pull_d']
vars_fecha_presentes = [v for v in vars_fecha if v in df.columns]

for col in vars_fecha_presentes:
    df[col] = pd.to_datetime(df[col], format='%b-%Y', errors='coerce')
    df[col] = (
        (df[col].dt.year  - FECHA_ORIGEN.year)  * 12 +
        (df[col].dt.month - FECHA_ORIGEN.month)
    ).astype('Int64')

print(f'Fechas convertidas a meses desde {FECHA_ORIGEN.date()}:')
for col in vars_fecha_presentes:
    print(f'  {col:<25} | nulos: {df[col].isnull().sum()} | '
          f'min: {df[col].min()} | max: {df[col].max()}')

print(f'\nDimensiones del dataset: {df.shape[0]:,} filas  |  {df.shape[1]} columnas')

### 2.7 Visualización de las variables restantes

Exploramos la distribución de las variables que hemos decidido mantener, 
comparando buenos y malos pagadores.

In [ ]:
# Variables numericas
vars_num = df.select_dtypes(include=['int64','float64']).columns.tolist()
if 'default' in vars_num:
    vars_num.remove('default')

n_cols = 4
n_rows = -(-len(vars_num) // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for i, var in enumerate(vars_num):
    ax = axes[i]
    for clase, color in PALETA_CLASES.items():
        datos = df.loc[df['default'] == clase, var].dropna()
        if not datos.empty:
            p99 = datos.quantile(0.99)
            datos = datos[datos <= p99]
        ax.hist(datos, bins=40, alpha=0.6, color=color,
                label=f'default={clase}', density=True, edgecolor='none')

    ax.set_title(var, fontweight='bold', fontsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('Densidad')
    ax.legend(fontsize=9)
    ax.grid(axis='y', linestyle='--', alpha=0.7)

for j in range(len(vars_num), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribucion de variables numericas por clase (default 0 vs 1)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '02_distribucion_numericas.png'), bbox_inches='tight')
plt.show()

In [ ]:
# Variables categoricas — Tasa de incumplimiento
def graficar_tasa_incumplimiento(df, variable, ax, top_n=10):
    """Grafica la tasa de incumplimiento por categoria de una variable."""
    tasa = (
        df.groupby(variable)['default']
          .agg(['mean', 'count'])
          .rename(columns={'mean': 'tasa_incumplimiento', 'count': 'n'})
          .sort_values('tasa_incumplimiento', ascending=True)
          .tail(top_n)
    )

    colores = plt.cm.RdYlGn_r(
        (tasa['tasa_incumplimiento'] - tasa['tasa_incumplimiento'].min()) /
        (tasa['tasa_incumplimiento'].max() - tasa['tasa_incumplimiento'].min() + 1e-9)
    )

    bars = ax.barh(tasa.index.astype(str), tasa['tasa_incumplimiento'] * 100,
                   color=colores, edgecolor='white', linewidth=0.5)

    for bar, (_, row) in zip(bars, tasa.iterrows()):
        ax.text(bar.get_width() + 0.3,
                bar.get_y() + bar.get_height() / 2,
                f"{row['tasa_incumplimiento']*100:.1f}%  (n={row['n']:,})",
                va='center', fontsize=10)

    ax.set_xlabel('Tasa de incumplimiento (%)')
    ax.set_title(f'Top {top_n}: {variable}', fontweight='bold')
    ax.axvline(df['default'].mean() * 100, color='navy', linestyle='--',
               linewidth=1.5, label=f'Media global ({df["default"].mean()*100:.1f}%)')
    ax.legend(fontsize=9)
    ax.set_xlim(0, tasa['tasa_incumplimiento'].max() * 100 * 1.45)


vars_categoricas = df.select_dtypes(include=['object', 'category']).columns.tolist()
exclude_cols = ['loan_status', 'default', 'issue_d', 'earliest_cr_line',
                'last_pymnt_d', 'next_pymnt_d', 'last_credit_pull_d', 'url', 'desc']
vars_cat = [v for v in vars_categoricas if v not in exclude_cols]

print(f'Variables categoricas a visualizar ({len(vars_cat)}): {vars_cat}')

n_cols_cat = 3
n_rows_cat = -(-len(vars_cat) // n_cols_cat)

fig, axes = plt.subplots(n_rows_cat, n_cols_cat, figsize=(22, n_rows_cat * 5))
axes = axes.flatten()

for i, var in enumerate(vars_cat):
    graficar_tasa_incumplimiento(df, var, axes[i], top_n=10)

for j in range(len(vars_cat), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Tasa de incumplimiento por variable categorica (Top 10)',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '04_tasa_incumplimiento_categoricas_grid.png'), bbox_inches='tight')
plt.show()

## 3. Tratamiento de valores faltantes

Se imputan los valores nulos por tipo de variable:
- **Numéricas:** mediana (robusta a outliers)
- **Categóricas:** moda (valor más frecuente)

In [ ]:
vars_numericas   = df.select_dtypes(include=['int64','float64']).columns.tolist()
vars_categoricas = df.select_dtypes(include=['object']).columns.tolist()

# No imputar la variable objetivo
vars_numericas   = [v for v in vars_numericas   if v != 'default']
vars_categoricas = [v for v in vars_categoricas if v != 'default']

# Imputacion de variables numericas con mediana
for col in vars_numericas:
    if df[col].isnull().any():
        mediana = df[col].median()
        df[col].fillna(mediana, inplace=True)

# Imputacion de variables categoricas con moda
for col in vars_categoricas:
    if df[col].isnull().any():
        moda = df[col].mode()[0]
        df[col].fillna(moda, inplace=True)

nulos_restantes = df.isnull().sum().sum()
print(f'Valores faltantes tratados.')
print(f'  Numericas imputadas con mediana   : {len(vars_numericas)} variables')
print(f'  Categoricas imputadas con moda    : {len(vars_categoricas)} variables')
print(f'  Valores faltantes restantes       : {nulos_restantes}')

## 4. Selección de variables con Information Value (IV)

El Information Value mide el poder predictivo de cada variable respecto al target.
Se utiliza el umbral convencional de IV >= 0.02 para seleccionar las variables relevantes.

In [ ]:
# Calculo de los bins WOE y del IV
bins = sc.woebin(df, y='default', check_cate_num=False, print_step=0)

# Extraccion del IV por variable
iv_df = pd.DataFrame([
    {'variable': var, 'iv': bin_df['total_iv'].iloc[0]}
    for var, bin_df in bins.items()
]).sort_values('iv', ascending=False)

print('Information Value por variable:')
print(iv_df.to_string())

In [ ]:
# Filtrado por IV >= 0.02
vars_seleccionadas = iv_df[iv_df['iv'] >= 0.02]['variable'].tolist()
print(f'Variables seleccionadas (IV >= 0.02): {len(vars_seleccionadas)}')

# Creacion del dataset filtrado
cols_finales = vars_seleccionadas + ['default']
cols_finales = [c for c in cols_finales if c in df.columns]
df_filtrado  = df[cols_finales].copy()

print(f'df_filtrado creado: {df_filtrado.shape}')

# Transformacion WOE
df_woe = sc.woebin_ply(df_filtrado, bins)

print(f'df_woe creado: {df_woe.shape}')

In [ ]:
# Visualizacion del Information Value
fig, ax = plt.subplots(figsize=(10, max(6, len(iv_df) * 0.35)))

colores = [
    '#F44336' if v > 0.3  else
    '#FF9800' if v > 0.1  else
    '#4CAF50' if v > 0.02 else
    '#9E9E9E'
    for v in iv_df['iv']
]

iv_ordenado = iv_df.sort_values('iv', ascending=True)

bars = ax.barh(iv_ordenado['variable'], iv_ordenado['iv'],
               color=colores[::-1], edgecolor='white')

for umbral, etiqueta, color in [
    (0.02, 'No util (<0.02)',  '#9E9E9E'),
    (0.1,  'Debil (0.1)',     '#4CAF50'),
    (0.3,  'Fuerte (0.3)',    '#F44336')
]:
    ax.axvline(umbral, linestyle='--', linewidth=1.2,
               color=color, alpha=0.7, label=etiqueta)

ax.set_xlabel('Information Value')
ax.set_title('Poder predictivo de cada variable (IV)', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '08_information_value.png'), bbox_inches='tight')
plt.show()

### 4.1 Distribución de variables seleccionadas

Se compara la distribución entre buenos y malos pagadores para las variables seleccionadas por IV.

In [ ]:
vars_numericas_filtradas = df_filtrado.select_dtypes(include=['int64','float64']).columns.tolist()
vars_categoricas_filtradas = df_filtrado.select_dtypes(include=['object']).columns.tolist()

if 'default' in vars_numericas_filtradas:
    vars_numericas_filtradas.remove('default')
if 'default' in vars_categoricas_filtradas:
    vars_categoricas_filtradas.remove('default')

print(f'Variables numericas seleccionadas: {len(vars_numericas_filtradas)}')
print(f'Variables categoricas seleccionadas: {len(vars_categoricas_filtradas)}')

In [ ]:
# Distribucion de variables numericas seleccionadas
for col in vars_numericas_filtradas:
    fig = plt.figure(figsize=(10, 6))
    sns.kdeplot(data=df_filtrado, x=col, hue='default', fill=True,
                common_norm=False, palette='viridis')
    plt.title(f'Distribucion de {col} por estado de incumplimiento')
    plt.xlabel(col)
    plt.ylabel('Densidad')
    plt.legend(title='Default', labels=['No Default (0)', 'Default (1)'])
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_PATH, f'dist_{col}_numerical.png'), bbox_inches='tight')
    plt.show()

In [ ]:
# Distribucion de variables categoricas seleccionadas
for col in vars_categoricas_filtradas:
    fig = plt.figure(figsize=(12, 7))
    sns.countplot(data=df_filtrado, x=col, hue='default', palette='viridis')
    plt.title(f'Conteo de {col} por estado de incumplimiento')
    plt.xlabel(col)
    plt.ylabel('Conteo')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Default', labels=['No Default (0)', 'Default (1)'])
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_PATH, f'dist_{col}_categorical.png'), bbox_inches='tight')
    plt.show()

## 5. Manejo del desbalance de clases

Se aplica una estrategia de remuestreo dependiendo de la severidad del desbalance:
- Si la tasa de incumplimiento es inferior al 15%, se utiliza **SMOTE** (sobremuestreo sintético).
- En caso contrario, se aplica **RandomUnderSampler** (submuestreo).

In [ ]:
X = df_woe.drop(columns=['default'])
y = df_woe['default']

tasa_original = y.mean()
print(f'Tasa de incumplimiento original: {tasa_original:.1%}')
print(f'  Buenos pagadores : {(y==0).sum():,}')
print(f'  Malos pagadores  : {(y==1).sum():,}')

if tasa_original < 0.15:
    print('\nAplicando SMOTE (desbalance severo)...')
    remuestreo = SMOTE(random_state=42)
    estrategia = 'SMOTE'
else:
    print('\nAplicando RandomUnderSampler (desbalance moderado)...')
    remuestreo = RandomUnderSampler(random_state=42)
    estrategia = 'UnderSampling'

X_res, y_res = remuestreo.fit_resample(X, y)

print(f'\nRemuestreo ({estrategia}) aplicado:')
print(f'  Buenos pagadores : {(y_res==0).sum():,}')
print(f'  Malos pagadores  : {(y_res==1).sum():,}')
print(f'  Nueva tasa       : {y_res.mean():.1%}')

## 6. División train / validation / test

Se separan los datos en tres conjuntos estratificados:
- **70%** entrenamiento
- **15%** validación
- **15%** prueba

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X_res, y_res, test_size=0.30, random_state=42, stratify=y_res
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print('Division train / validation / test:')
print(f'  Train      : {len(X_train):,} filas  ({len(X_train)/len(X_res)*100:.0f}%)')
print(f'  Validation : {len(X_val):,} filas  ({len(X_val)/len(X_res)*100:.0f}%)')
print(f'  Test       : {len(X_test):,} filas  ({len(X_test)/len(X_res)*100:.0f}%)')
print(f'\nTasa de incumplimiento:')
print(f'  Train      : {y_train.mean():.1%}')
print(f'  Validation : {y_val.mean():.1%}')
print(f'  Test       : {y_test.mean():.1%}')

## 7. Guardado de resultados

In [ ]:
import joblib

# Guardado de los splits
splits = {
    'X_train': X_train, 'y_train': y_train,
    'X_val'  : X_val,   'y_val'  : y_val,
    'X_test' : X_test,  'y_test' : y_test
}
joblib.dump(splits, os.path.join(OUTPUT_PATH, 'splits.pkl'))

# Guardado del dataset filtrado (antes del remuestreo) para la scorecard
df_filtrado.to_parquet(os.path.join(DATA_PATH, 'loan_procesado.parquet'), index=False)

print('Archivos guardados:')
print(f'  splits.pkl             -> {OUTPUT_PATH}')
print(f'  loan_procesado.parquet -> {DATA_PATH}')

print(f'\nDimensiones de X_train: {X_train.shape}')

## Resumen

| Categoría de variables eliminadas | Criterio | Impacto si se incluyen |
|---|---|---|
| Identificadores | Sin información predictiva | Ruido |
| Alto % de nulos | > 40% de valores faltantes | Sesgo por imputación |
| Varianza cero | Un único valor | Sin poder discriminante |
| Alta cardinalidad | > 500 valores únicos | Sobreajuste |
| Data leakage | Información posterior al evento | Optimismo inflado, modelo inútil |
| Variable origen | Fuente directa de `default` | Leakage perfecto |

**Pasos realizados:**

1. Eliminación de variables no útiles (identificadores, alta nulidad, varianza cero, alta cardinalidad, data leakage).
2. Conversión de fechas a escala numérica lineal.
3. Imputación de valores faltantes (mediana para numéricas, moda para categóricas).
4. Selección de variables mediante Information Value (IV >= 0.02).
5. Transformación WOE de las variables seleccionadas.
6. Tratamiento del desbalance de clases.
7. División en conjuntos de entrenamiento (70%), validación (15%) y prueba (15%).